# Working with real data: cleaning, windows and performance

This notebook continues from `01_intro_pyspark.ipynb`. Instead of building a DataFrame by hand, it reads a CSV file of e-commerce orders from `data/input/sales.csv` and walks through the operations you'll use most often on real datasets:

1. Reading a file with an explicit schema
2. Handling missing values
3. Working with dates
4. Pivot tables
5. Window functions (rankings, running totals, comparisons with the previous period)
6. Lazy evaluation, `explain()` and caching
7. Writing partitioned Parquet

## 1. Setup

As in the first notebook, all imports go in the first cell.

In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
from pyspark.sql.window import Window

In [ ]:
BASE_FILEPATH = Path("/home/jovyan/data")
INPUT_FILEPATH = BASE_FILEPATH / "input"
OUTPUT_FILEPATH = BASE_FILEPATH / "output"

In [ ]:
appName = "DataFrameAdvanced"

spark = SparkSession.builder \
    .appName(appName) \
    .getOrCreate()

spark_ui_port = spark.sparkContext.uiWebUrl.rsplit(":", 1)[-1]
print(f"Spark UI: http://localhost:{spark_ui_port}")

## 2. Reading a CSV with an explicit schema

The file has one row per order. Declaring the schema lets Spark parse `order_date` directly as a date and the numeric columns with the right types, without the extra pass over the data that `inferSchema` requires.

In [ ]:
sales_schema = StructType(fields=[
    StructField(name="order_id", dataType=IntegerType(), nullable=False),
    StructField(name="order_date", dataType=DateType(), nullable=False),
    StructField(name="customer_id", dataType=StringType(), nullable=False),
    StructField(name="city", dataType=StringType(), nullable=True),
    StructField(name="category", dataType=StringType(), nullable=False),
    StructField(name="product", dataType=StringType(), nullable=False),
    StructField(name="quantity", dataType=IntegerType(), nullable=True),
    StructField(name="unit_price", dataType=DoubleType(), nullable=False),
])

sales_path = str(INPUT_FILEPATH / "sales.csv")

df_raw = spark.read \
    .schema(sales_schema) \
    .option(key="header", value=True) \
    .csv(path=sales_path)

df_raw.printSchema()
df_raw.show(n=5)

In [ ]:
print(f"Rows: {df_raw.count()}")

## 3. Handling missing values

Real data is rarely complete. In CSV files an empty field is read as `null`. First we count the `null` values in each column:

In [ ]:
df_raw.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_raw.columns
]).show()

There are two common strategies:

- **`na.drop`**: removes the rows with `null` values in the given columns. Here we use it for `quantity`, because without it we cannot compute the order revenue.
- **`fillna`**: replaces `null` values with a default. Here we use it for `city`, so the order is kept but marked as `Unknown`.

In [ ]:
df = df_raw \
    .na.drop(subset=["quantity"]) \
    .fillna(value="Unknown", subset=["city"])

print(f"Rows before: {df_raw.count()}, after: {df.count()}")

Then we add the `revenue` column for each order:

In [ ]:
df = df.withColumn(
    colName="revenue",
    col=F.round(F.col("quantity") * F.col("unit_price"), scale=2),
)

df.show(n=5)

## 4. Working with dates

The `pyspark.sql.functions` module provides many functions for dates: `year`, `month`, `dayofweek`, `date_format`, `datediff`, `date_add`, ... Here we extract the month (in `yyyy-MM` format, which sorts correctly as a string) and the day of the week:

In [ ]:
df = df \
    .withColumn(colName="month", col=F.date_format(F.col("order_date"), "yyyy-MM")) \
    .withColumn(colName="weekday", col=F.date_format(F.col("order_date"), "EEEE"))

df.select("order_id", "order_date", "month", "weekday").show(n=5)

Monthly revenue:

In [ ]:
df_monthly = df \
    .groupBy("month") \
    .agg(
        F.count("order_id").alias("orders"),
        F.round(F.sum("revenue"), 2).alias("revenue"),
    ) \
    .orderBy("month")

df_monthly.show()

## 5. Pivot tables

`pivot` turns the distinct values of a column into new columns, like a pivot table in Excel. Here we compute the revenue for each month (rows) and category (columns):

In [ ]:
df.groupBy("month") \
    .pivot("category") \
    .agg(F.round(F.sum("revenue"), 2)) \
    .orderBy("month") \
    .show()

## 6. Window functions

An aggregation with `groupBy` returns **one row per group**. A **window function** instead computes a value over a group of rows (the *window*) but keeps **every row**. A window is defined with:

- **`partitionBy`**: how rows are grouped (like `groupBy`)
- **`orderBy`**: how rows are sorted within each group
- optionally **`rowsBetween`**: which rows, relative to the current one, are included in the computation

### Ranking

Which are the best-selling products in each category? First we compute the revenue per product, then we rank the products within their category:

In [ ]:
df_products = df \
    .groupBy("category", "product") \
    .agg(F.round(F.sum("revenue"), 2).alias("revenue"))

window_by_category = Window \
    .partitionBy("category") \
    .orderBy(F.desc("revenue"))

df_products \
    .withColumn(colName="rank", col=F.rank().over(window_by_category)) \
    .orderBy("category", "rank") \
    .show()

A typical use is the **top N per group**: here we keep only the best-selling product of each category.

In [ ]:
df_products \
    .withColumn(colName="rank", col=F.rank().over(window_by_category)) \
    .filter(condition=F.col("rank") == 1) \
    .drop("rank") \
    .show()

### Comparison with the previous period

`lag` reads the value of a previous row in the window (`lead` reads a following one). Here we compare each month's revenue with the previous month's. The window has no `partitionBy`, so Spark moves all the rows to a single partition and prints a warning: that's fine here, because there are only 12 rows.

In [ ]:
window_by_month = Window.orderBy("month")

df_monthly \
    .withColumn(colName="prev_revenue", col=F.lag("revenue", 1).over(window_by_month)) \
    .withColumn(
        colName="change_pct",
        col=F.round((F.col("revenue") - F.col("prev_revenue")) / F.col("prev_revenue") * 100, scale=1),
    ) \
    .show()

### Running total

With `rowsBetween(Window.unboundedPreceding, Window.currentRow)` the window includes all rows from the start of the partition up to the current one: summing over it gives the **running total**. Here we compute the cumulative revenue of each city over the months:

In [ ]:
df_city_monthly = df \
    .groupBy("city", "month") \
    .agg(F.round(F.sum("revenue"), 2).alias("revenue"))

window_running = Window \
    .partitionBy("city") \
    .orderBy("month") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_city_monthly \
    .withColumn(colName="cumulative_revenue", col=F.round(F.sum("revenue").over(window_running), scale=2)) \
    .filter(condition=F.col("city") == "Roma") \
    .orderBy("month") \
    .show()

## 7. Lazy evaluation, `explain()` and caching

Spark transformations (`select`, `filter`, `groupBy`, `withColumn`, ...) are **lazy**: they don't compute anything, they only build an execution plan. Computation starts only when an **action** is called (`show`, `count`, `collect`, `write`, ...).

`explain()` shows the plan Spark will run. It's read bottom-up: first the file is read (`FileScan csv`), then the transformations are applied.

In [ ]:
df_top_cities = df \
    .filter(condition=F.col("category") == "Electronics") \
    .groupBy("city") \
    .agg(F.round(F.sum("revenue"), 2).alias("revenue")) \
    .orderBy(F.desc("revenue"))

df_top_cities.explain()

Every action recomputes the whole plan from the beginning, including reading the CSV. When the same DataFrame is used many times, it can be kept in memory with **`cache()`**. Caching is lazy too: the data is stored the first time an action is run.

> In the Spark UI ([http://localhost:4040](http://localhost:4040)), the **Storage** tab shows the cached DataFrames and how much memory they use.

In [ ]:
df.cache()
df.count()  # first action: reads the CSV and fills the cache

print(f"Cached: {df.is_cached}")

From now on, actions on `df` read the data from memory. When it's no longer needed, free the memory with `unpersist()`:

In [ ]:
df_top_cities.show()

df.unpersist()

## 8. Writing partitioned Parquet

With `partitionBy`, Spark writes a subfolder for each value of the column (`category=Books/`, `category=Electronics/`, ...). A query that filters on that column reads only the matching folders: this is called **partition pruning**, and it's one of the most effective optimizations on large datasets.

In [ ]:
partitioned_path = str(OUTPUT_FILEPATH / "sales_by_category")

df.write \
    .mode("overwrite") \
    .partitionBy("category") \
    .parquet(path=partitioned_path)

sorted(p.name for p in Path(partitioned_path).iterdir() if p.is_dir())

When reading with a filter on `category`, the plan shows the condition among the `PartitionFilters`: Spark doesn't even open the files of the other categories.

In [ ]:
df_books = spark.read \
    .parquet(partitioned_path) \
    .filter(condition=F.col("category") == "Books")

df_books.explain()
df_books.show(n=5)

## 9. All done!

**Next steps:**
- `03_mllib.ipynb`: training a Machine Learning model with Spark MLlib
- Try other window functions: `dense_rank`, `row_number`, `ntile`, `lead`
- Look at the **SQL / DataFrame** tab in the Spark UI to see the plans of the queries you just ran

When done working, close the SparkSession to release resources:

In [ ]:
spark.stop()